# BugLens — fine-tune CodeRankEmbed on Kaggle (FR-T3)

One command, on a free Kaggle GPU (T4/P100, 16 GB). Steps:

1. **Settings → Accelerator → GPU T4 x2 (or P100)**, Internet **on**.
2. Add two Kaggle datasets as inputs (Add Data → Upload):
   - `buglens-src`: the project folder (zip of the repo without `data/`),
   - `buglens-pairs`: `data/pairs/train.parquet` and `data/pairs/val.parquet` from `buglens pairs`.
3. Run all cells. Download `coderankembed-ft.zip` from the Output tab and unzip it into
   `data/models/coderankembed-ft/` on your machine (or upload it to the HF Hub).

Training itself is the same code as `buglens finetune` locally (`src/buglens/training/finetune.py`).

In [ ]:
import glob, os, shutil, subprocess
SRC = glob.glob('/kaggle/input/buglens-src*')[0]
PAIRS = glob.glob('/kaggle/input/buglens-pairs*')[0]
WORK = '/kaggle/working/buglens'
if not os.path.exists(WORK):
    shutil.copytree(SRC, WORK)
os.makedirs(f'{WORK}/data/pairs', exist_ok=True)
for f in glob.glob(f'{PAIRS}/**/*.parquet', recursive=True):
    shutil.copy(f, f'{WORK}/data/pairs/')
print(os.listdir(f'{WORK}/data/pairs'))

In [ ]:
# Kaggle images already ship torch + CUDA; install the rest of BugLens without touching torch.
!pip install -q --no-deps -e /kaggle/working/buglens
!pip install -q "sentence-transformers>=3.0" einops datasets pyarrow typer rich pydantic pyyaml networkx unidiff bm25s faiss-cpu lightgbm graphviz google-genai

In [ ]:
# Same config, but fp16 on (fast on T4/P100 tensor cores).
import yaml
cfg = yaml.safe_load(open(f'{WORK}/configs/default.yaml'))
cfg['training']['fp16'] = True
yaml.safe_dump(cfg, open(f'{WORK}/configs/kaggle.yaml', 'w'))

In [ ]:
%cd /kaggle/working/buglens
!buglens finetune --config configs/kaggle.yaml

In [ ]:
shutil.make_archive('/kaggle/working/coderankembed-ft', 'zip', f'{WORK}/data/models/coderankembed-ft')
shutil.copy(f'{WORK}/reports/figures/finetune_loss.png', '/kaggle/working/')
shutil.copy(f'{WORK}/reports/finetune_eval.json', '/kaggle/working/')
print(open('/kaggle/working/finetune_eval.json').read()[:1500])